# فاز اول: توصیف زیرمجموعهٔ NACC UDS + MRI

این دفترچه اجرای تحلیل توصیفی و ممیزی داده را از روی CSVهای محلی بازتولید می‌کند. واحد تحلیل فرد یکتا (`NACCID`) است. هیچ آزمون استنباطی در این فاز اجرا نمی‌شود.


In [1]:
from pathlib import Path
import json, subprocess, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "udslong_imputed.csv").exists())
subprocess.run([sys.executable, str(ROOT / "src" / "phase1_analysis.py")], cwd=ROOT, check=True, capture_output=True, text=True)
results = json.loads((ROOT / "reports" / "phase1" / "phase1_results.json").read_text(encoding="utf-8"))
print("Analysis complete. Joined participants:", results["linkage"]["common_participants"])
print("Unmatched UDS/MRI IDs:", results["linkage"]["unmatched_UDS"], results["linkage"]["unmatched_MRI"])


Analysis complete. Joined participants: 1237
Unmatched UDS/MRI IDs: 0 0


## نمونه و متغیرهای کمی اصلی

چارک‌ها با قاعدهٔ خطی محاسبه شده‌اند. SD انحراف معیار نمونه است. مقادیر MOCATOTS خارج از بازهٔ معتبر ۰–۳۰ حذف می‌شوند؛ دادهٔ جایگذاری‌شده همچنان جایگذاری‌شده باقی می‌ماند و این کد روش آن را حدس نمی‌زند.


In [2]:
for name, s in results["summary_statistics"].items():
    print(f"{name}: n={s['n']}, mean={s.get('mean'):.3f}, SD={s.get('sd'):.3f}, median={s.get('median'):.3f}, Q1={s.get('q1'):.3f}, Q3={s.get('q3'):.3f}, range=({s.get('min'):.3f}, {s.get('max'):.3f})")


NACCAGE: n=1237, mean=68.192, SD=8.039, median=68.000, Q1=63.000, Q3=74.000, range=(41.000, 91.000)
EDUC_years: n=1235, mean=16.257, SD=2.719, median=16.000, Q1=14.000, Q3=18.000, range=(0.000, 25.000)
MOCATOTS_valid_0_30: n=1228, mean=24.393, SD=4.425, median=25.400, Q1=23.000, Q3=27.000, range=(0.000, 30.000)
CDRSUM: n=1237, mean=0.822, SD=1.723, median=0.000, Q1=0.000, Q3=1.000, range=(0.000, 15.000)
ENTORHINAL_BILATERAL_AVGTH_mm: n=1237, mean=3.014, SD=0.356, median=3.058, Q1=2.803, Q3=3.252, range=(1.214, 3.925)
LH_ENTORHINAL_AVGTH_mm: n=1237, mean=2.970, SD=0.384, median=3.021, Q1=2.763, Q3=3.228, range=(0.878, 3.979)
RH_ENTORHINAL_AVGTH_mm: n=1237, mean=3.058, SD=0.401, median=3.097, Q1=2.852, Q3=3.318, range=(0.729, 3.967)


## مقایسهٔ توصیفی بر اساس NACCUDSD

کدهای NACCUDSD: ۱ طبیعی، ۲ اختلال بدون MCI، ۳ MCI و ۴ دمانس. کد ۲ تنها ۷ فرد دارد؛ هر خلاصهٔ آن باید با احتیاط تفسیر شود. گروه‌بندی در اینجا جایگزین تشخیص علت‌شناختی آلزایمر نیست.


In [3]:
for code, group in results["by_NACCUDSD"].items():
    m, t = group["MOCATOTS"], group["ENTORHINAL_BILATERAL_AVGTH_mm"]
    print(f"{code} {group['label']}: n={group['n']}; valid MoCA n={m['n']}, median={m['median']:.2f}, IQR=({m['q1']:.2f}, {m['q3']:.2f}); thickness median={t['median']:.3f} mm")


1 Normal cognition: n=847; valid MoCA n=842, median=26.20, IQR=(25.00, 28.00); thickness median=3.103 mm
2 Impaired, not MCI: n=7; valid MoCA n=7, median=26.80, IQR=(25.80, 27.70); thickness median=3.188 mm
3 MCI: n=252; valid MoCA n=251, median=23.00, IQR=(20.00, 25.00); thickness median=2.966 mm
4 Dementia: n=131; valid MoCA n=128, median=17.00, IQR=(13.00, 21.00); thickness median=2.691 mm


## ممیزی پیوند و کیفیت


In [4]:
print("Linkage:", results["linkage"])
print("MOCATOTS invalid values:", results["file_quality"]["MOCATOTS_nonanalytic_values"])
print("Valid fractional MOCATOTS values:", results["file_quality"]["MOCATOTS_valid_fractional_n"])
print("Integer-only sensitivity summary (not an observed-case sample):", results["file_quality"]["MOCATOTS_integer_sensitivity_summary"])
print("MRI nonpositive volume cells:", results["file_quality"]["MRI_measure_audit"]["nonpositive_volume_cells"])
print("Adjacency matrix audit:", results["adjacency_file"])


Linkage: {'UDS_rows': 1237, 'MRI_rows': 1237, 'unique_UDS_NACCID': 1237, 'unique_MRI_NACCID': 1237, 'common_participants': 1237, 'unmatched_UDS': 0, 'unmatched_MRI': 0, 'duplicate_UDS_ID_rows': 0, 'duplicate_MRI_ID_rows': 0, 'NACCUDSD_disagreements': 0, 'participants_with_MRI_entorhinal_thickness': 1237, 'participants_without_valid_MRI_entorhinal_thickness': 0, 'participants_with_both_MRI_and_UDS_valid_main_variables': 1228}
MOCATOTS invalid values: {'39.4': 2, '38.8': 1, '34.2': 1, '88.0': 4, '30.8': 1}
Valid fractional MOCATOTS values: 244
Integer-only sensitivity summary (not an observed-case sample): {'n': 984, 'mean': 24.028455284552845, 'sd': 4.739308066390187, 'min': 0.0, 'q1': 22.0, 'median': 25.0, 'q3': 27.0, 'iqr': 5.0, 'max': 30.0}
MRI nonpositive volume cells: [{'variable': 'LH_ENTORHINAL_GVOL', 'count_nonpositive': 1}, {'variable': 'LH_POSTERIORCINGULATE_GVOL', 'count_nonpositive': 1}]
Adjacency matrix audit: {'n_regions': 62, 'row_column_labels_match': True, 'symmetric': 

## نمودارها و گزارش

نُه شکل SVG با محورهای عنوان‌دار در `figures/phase1/` تولید می‌شوند. PDF فارسی در دو مسیر `reports/phase1.pdf` و `reports/phase1/report_phase1.pdf` ذخیره می‌شود. برای بازتولید PDF، بسته‌های `requirements.txt` و ابزار راست‌به‌چپ `fa_pdf.py` محیط GapWork لازم است. ورودی‌ها تاریخ/شناسهٔ مراجعه و اسکن ندارند؛ بنابراین زمان‌هم‌ترازی UDS و MRI از اتصال شناسه استنباط نمی‌شود.


In [5]:
print("Figures:")
for path in results["figure_files"]: print(" -", path)
print("Report:", ROOT / "reports" / "phase1.pdf")


Figures:
 - figures/phase1/01_age_distribution.svg
 - figures/phase1/02_moca_distribution.svg
 - figures/phase1/03_entorhinal_distribution.svg
 - figures/phase1/04_cognitive_status.svg
 - figures/phase1/05_sex_distribution.svg
 - figures/phase1/06_moca_by_status.svg
 - figures/phase1/07_thickness_by_status.svg
 - figures/phase1/08_moca_thickness_scatter.svg
 - figures/phase1/09_linkage.svg
Report: /Users/benjamin/Documents/Statistic Project/Statistics_Project/reports/phase1.pdf
